# 05 - Preset Tuning for GPU Budget

This notebook explains how to tune memory/speed knobs without changing math semantics.

Preset intent:
- `cpu`: maximum compatibility and CPU execution
- `gpu`: keep memory and compute on CUDA for the tutorial default
- `hybrid`: available when you want CPU memory with CUDA compute


In [1]:
# EN: This notebook only needs the preset resolver and the Circuit builder, because the focus is configuration rather than training.
# KO: 이 노트북은 학습보다 설정 비교가 중심이므로 preset resolver와 Circuit 빌더 위주로만 불러옵니다.

import warnings
warnings.filterwarnings("ignore")

from padopauli import resolve_preset, Circuit

In [2]:
base = resolve_preset("gpu")
print("base:", base)

custom = resolve_preset(
    "gpu",
    overrides={
        "max_weight": 6,
        "dtype": "float32",
    },
)
print("custom:", custom)

base: ExecutionPreset(memory_device='cuda', compute_device='cuda', dtype='float64', max_weight=1000000000, weight_x=1.0, weight_y=1.0, weight_z=1.0, chunk_size=10000000000)
custom: ExecutionPreset(memory_device='cuda', compute_device='cuda', dtype='float32', max_weight=6, weight_x=1.0, weight_y=1.0, weight_z=1.0, chunk_size=10000000000)


## Example compile with targeted overrides

Common knobs (see `resolve_preset`):
- `max_weight`, `weight_x`/`weight_y`/`weight_z`: structural truncation strength (max Pauli weight kept, per-axis weighting)
- `dtype`, `chunk_size`: numeric precision and compute chunking
- `memory_device`/`compute_device`: storage vs compute placement — this is what actually distinguishes the `cpu`/`gpu`/`hybrid` presets

`Circuit.compile(...)` exposes `max_weight`, `weight_x/y/z`, `dtype`, and `chunk_size` directly as keyword arguments. `memory_device`/`compute_device` are chosen via the `preset` name itself (`cpu`/`gpu`/`hybrid`); to override them individually, call `resolve_preset(...)` or `compile_program(..., preset_overrides=...)` directly (`preset_overrides` is the full low-level surface).


In [3]:
qc = Circuit(n_qubits=2)
qc.rzz(0, 1, param_idx=0)

program = qc.compile(
    observables=[("ZZ", [0, 1])],
    preset="gpu",
    max_weight=6,
)

print("program preset:", program.preset)

propagate:   0%|          | 0/1 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 1


zero-filter:   0%|          | 0/1 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 1 (100.000000% of peak)
program preset: ExecutionPreset(memory_device='cuda', compute_device='cuda', dtype='float64', max_weight=6, weight_x=1.0, weight_y=1.0, weight_z=1.0, chunk_size=10000000000)


## Practical policy
1. Start from `gpu` defaults.
2. If memory is tight, reduce `max_weight` (and/or `weight_x/y/z`) first — this caps propagated term count directly.
3. If speed is too low and memory allows, move `memory_device` closer to `compute_device` (e.g. `gpu` preset instead of `hybrid`) or raise `chunk_size`.
4. Always re-check error vs exact small-n baseline after tuning.